# 01 · Groq Basics

**Goal:** learn the primitives CareerForge is built on — installing the Groq SDK,
initializing the client from an environment variable, running chat completions,
using system prompts to steer behavior, and forcing **strict JSON** output.

> These same helpers reappear (hardened) in `02_career_forge_agent/utils.py`.

## 1. Install the SDK

Run once per environment.

In [1]:
%pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\sharm\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Initialize the client from an environment variable

**Never hardcode keys in a notebook.** Put `GROQ_API_KEY=...` in a `.env` file
(git-ignored) and load it. `get_client()` fails loudly if the key is missing.

In [2]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()  # reads .env in the working directory

def get_client() -> Groq:
    key = os.getenv("GROQ_API_KEY")
    if not key:
        raise RuntimeError("GROQ_API_KEY is not set. Add it to your .env file.")
    return Groq(api_key=key)

client = get_client()
print("Groq client ready.")

Groq client ready.


## 3. A basic chat completion

The message list is OpenAI-style: a `system` message sets the persona/rules,
`user` messages carry the request. We route to the fast model here.

In [3]:
FAST_MODEL = "llama-3.1-8b-instant"
DEEP_MODEL = "llama-3.3-70b-versatile"

resp = client.chat.completions.create(
    model=FAST_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise career coach."},
        {"role": "user", "content": "In one sentence, why do system prompts matter?"},
    ],
    temperature=0.3,
)
print(resp.choices[0].message.content)

System prompts matter because they help tailor your responses to the specific requirements and constraints of the platform or tool you're interacting with, ensuring clarity, accuracy, and relevance in your communication.


## 4. System prompts change everything

Same question, two personas. Notice how the `system` message reshapes tone and
content without touching the user request.

In [4]:
def ask(system: str, user: str, model: str = FAST_MODEL) -> str:
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.4,
    )
    return r.choices[0].message.content

q = "Should I learn SQL?"
print("STRICT MENTOR:\n", ask("You are a blunt senior engineer. Be terse.", q))
print("\nENCOURAGING COACH:\n", ask("You are a warm, encouraging mentor.", q))

STRICT MENTOR:
 Yes. SQL is a fundamental skill for any data-related work. You'll need it for most jobs.

ENCOURAGING COACH:
 Learning SQL can be an incredibly valuable skill, and I'm happy to explain why.

SQL (Structured Query Language) is a fundamental language used for managing and analyzing data in relational databases. With SQL, you can:

1. **Extract insights**: SQL allows you to query and analyze data to gain valuable insights, making it an essential skill for data analysis, business intelligence, and decision-making.
2. **Work with databases**: SQL is used to interact with databases, which store and manage large amounts of data. Knowing SQL enables you to work with databases, create and manage tables, and perform various operations.
3. **Build data-driven applications**: SQL is a crucial skill for building data-driven applications, such as web applications, mobile apps, and data visualization tools.
4. **Enhance career prospects**: Knowing SQL can significantly boost your care

## 5. Strict JSON mode

For agentic pipelines we need **machine-readable** output, not prose. Groq
supports `response_format={"type": "json_object"}`. Always describe the exact
shape in the prompt and include the word "JSON".

In [5]:
import json

def json_completion(system: str, user: str, model: str = FAST_MODEL) -> dict:
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.1,
        response_format={"type": "json_object"},
    )
    return json.loads(r.choices[0].message.content)

system = (
    "You extract skills from text. Respond ONLY with JSON of the form "
    '{"skills": ["..."], "seniority": "junior|mid|senior"}.'
)
user = "5 years building FastAPI microservices, some Kubernetes, mentored 2 juniors."
data = json_completion(system, user)
print(type(data), data)
print("First skill:", data["skills"][0])

<class 'dict'> {'skills': ['FastAPI', 'Kubernetes', 'Mentoring'], 'seniority': 'mid'}
First skill: FastAPI


## 6. Validate JSON with Pydantic (preview of the real app)

Parsing raw dicts is fragile. In the app we validate every model response with a
Pydantic schema so bad output fails fast.

In [6]:
from typing import List, Literal
from pydantic import BaseModel, ValidationError

class Extraction(BaseModel):
    skills: List[str]
    seniority: Literal["junior", "mid", "senior"]

try:
    parsed = Extraction.model_validate(data)
    print("Validated:", parsed)
except ValidationError as e:
    print("Schema mismatch:", e)

Validated: skills=['FastAPI', 'Kubernetes', 'Mentoring'] seniority='mid'


### Recap
- Client is built from `GROQ_API_KEY` (never hardcoded).
- `system` messages steer behavior; `user` messages carry the task.
- `response_format={"type": "json_object"}` + a schema description = reliable structured output.
- Validate with Pydantic before trusting the data.

Next: **02_rag_foundations** — grounding prompts in retrieved context.